In [5]:
import sys
import os
import requests
import pyreadstat
import pandas as pd

# 1. Resolve Project Root Directory
current_dir = os.path.abspath("")
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "notebooks" else current_dir
nhanes_dir = os.path.join(project_root, "data", "raw", "nhanes")
os.makedirs(nhanes_dir, exist_ok=True)

# 2. Direct Official CDC CDN URL for DEMO_G
demo_filename = "DEMO_G.XPT"
demo_path = os.path.join(nhanes_dir, demo_filename)
cdc_url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/{demo_filename}"

# 3. Download if not exists or if file is incomplete
if not os.path.exists(demo_path) or os.path.getsize(demo_path) < 10000:
    print(f"Downloading {demo_filename} directly from CDC...")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(cdc_url, headers=headers, stream=True, timeout=60)
    response.raise_for_status()
    with open(demo_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded successfully: {demo_path}")

# 4. Read SAS XPT using pyreadstat
df, meta = pyreadstat.read_xport(demo_path, encoding="latin1")

# 5. Filter Adults (age >= 20)
selected_cols = ["SEQN", "RIDAGEYR", "RIAGENDR"]
adults_df = df[df["RIDAGEYR"] >= 20][selected_cols].copy()

print(f"File Size: {os.path.getsize(demo_path) / (1024*1024):.2f} MB")
print(f"Total Participants: {len(df)}")
print(f"Adults >= 20: {len(adults_df)}")
display(adults_df.head())

File Size: 3.58 MB
Total Participants: 9756
Adults >= 20: 5560


,SEQN,RIDAGEYR,RIAGENDR
0,62161.0,22.0,1.0
3,62164.0,44.0,2.0
8,62169.0,21.0,1.0
11,62172.0,43.0,2.0
13,62174.0,80.0,1.0


In [9]:
import sys
import os
import pandas as pd

current_dir = os.path.abspath("")
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "notebooks" else current_dir
if project_root not in sys.path:
    sys.path.append(project_root)

nhanes_dir = os.path.join(project_root, "data", "raw", "nhanes")
interim_dir = os.path.join(project_root, "data", "interim")
processed_dir = os.path.join(project_root, "data", "processed")

from src.data.wearable_processor import get_valid_wear_participants_from_paxday
from src.data.prepare_handover import create_clean_member2_handover

# 1. Load previously saved 2011-2014 clinical cohort
clinical_path = os.path.join(interim_dir, "nhanes_2011_2014_clinical_labels.parquet")
full_clinical_df = pd.read_parquet(clinical_path)

# 2. Download PAXDAY_G & PAXDAY_H and filter valid wear participants (>= 3 days, >= 600 mins)
print("Evaluating accelerometer wear compliance (PAXDAY_G & PAXDAY_H)...")
valid_wear_df = get_valid_wear_participants_from_paxday(
    nhanes_dir=nhanes_dir,
    min_wake_minutes=600,
    min_valid_days=3
)
print(f"Total NHANES participants meeting wear compliance: {len(valid_wear_df)}")

# 3. Create final leakage-free clean cohort for Member 2
clean_cohort_df = create_clean_member2_handover(
    clinical_df=full_clinical_df,
    valid_wear_df=valid_wear_df,
    processed_dir=processed_dir,
    interim_dir=interim_dir
)

print(f"\nFINAL CLEAN COHORT SAMPLE SIZE: {len(clean_cohort_df)}")
print("Columns passed to Member 2:")
print(list(clean_cohort_df.columns))
print("\nFinal Target Class Distribution:")
print(clean_cohort_df["metabolic_syndrome_target"].value_counts(normalize=True).round(3))
display(clean_cohort_df.head())

Evaluating accelerometer wear compliance (PAXDAY_G & PAXDAY_H)...
Total NHANES participants meeting wear compliance: 14606

FINAL CLEAN COHORT SAMPLE SIZE: 4604
Columns passed to Member 2:
['SEQN', 'survey_cycle', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'WTSAF2YR', 'valid_days_count', 'mean_daily_wear_mins', 'metabolic_syndrome_target', 'smoking_status']

Final Target Class Distribution:
metabolic_syndrome_target
0    0.569
1    0.431
Name: proportion, dtype: float64


,SEQN,survey_cycle,RIDAGEYR,RIAGENDR,RIDRETH3,WTSAF2YR,valid_days_count,mean_daily_wear_mins,metabolic_syndrome_target,smoking_status
0,62161.0,2011-2012,22.0,1.0,3.0,240011.713,9,2.949179e+06,0,never
1,62164.0,2011-2012,44.0,2.0,3.0,288182.780,9,2.541249e+06,0,never
2,62169.0,2011-2012,21.0,1.0,6.0,30061.013,9,2.468200e+06,0,never
3,62172.0,2011-2012,43.0,2.0,4.0,63279.908,9,4.783470e+06,0,current
4,62177.0,2011-2012,51.0,1.0,6.0,17583.539,9,2.635476e+06,0,current


In [12]:
import inspect
import src.data.wearable_processor as wp
import src.data.prepare_handover as ph

print("wearable_processor functions:", [name for name, obj in inspect.getmembers(wp, inspect.isfunction)])
print("prepare_handover functions:", [name for name, obj in inspect.getmembers(ph, inspect.isfunction)])

wearable_processor functions: ['download_nhanes_file', 'extract_filtered_minute_streams', 'get_valid_wear_participants_from_paxday']
prepare_handover functions: ['create_clean_member2_handover']


In [8]:
import sys
import os

current_dir = os.path.abspath("")
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "notebooks" else current_dir
if project_root not in sys.path:
    sys.path.append(project_root)

nhanes_dir = os.path.join(project_root, "data", "raw", "nhanes")
interim_dir = os.path.join(project_root, "data", "interim")
os.makedirs(interim_dir, exist_ok=True)

from src.data.build_cohort import build_combined_nhanes_clinical_cohort

# Build combined 2011-2014 NHANES clinical cohort
full_clinical_df = build_combined_nhanes_clinical_cohort(nhanes_dir)

# Save intermediate clinical labels table
interim_clinical_path = os.path.join(interim_dir, "nhanes_2011_2014_clinical_labels.parquet")
full_clinical_df.to_parquet(interim_clinical_path, index=False)

print(f"Combined 2011-2014 Analytical Sample Size: {len(full_clinical_df)}")
print("\nCounts by Survey Cycle:")
print(full_clinical_df["survey_cycle"].value_counts())
print("\nOverall Risk Target Distribution:")
print(full_clinical_df["metabolic_syndrome_target"].value_counts(normalize=True).round(3))
display(full_clinical_df[["SEQN", "survey_cycle", "RIDAGEYR", "RIAGENDR", "metabolic_syndrome_target"]].head())

Combined 2011-2014 Analytical Sample Size: 5295

Counts by Survey Cycle:
survey_cycle
2013-2014    2668
2011-2012    2627
Name: count, dtype: int64

Overall Risk Target Distribution:
metabolic_syndrome_target
0    0.578
1    0.422
Name: proportion, dtype: float64


,SEQN,survey_cycle,RIDAGEYR,RIAGENDR,metabolic_syndrome_target
0,62161.0,2011-2012,22.0,1.0,0
1,62164.0,2011-2012,44.0,2.0,0
2,62169.0,2011-2012,21.0,1.0,0
3,62172.0,2011-2012,43.0,2.0,0
4,62177.0,2011-2012,51.0,1.0,0


In [6]:
import sys
import os
import pandas as pd

# 1. Resolve Project Root & Import Modules
current_dir = os.path.abspath("")
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "notebooks" else current_dir
if project_root not in sys.path:
    sys.path.append(project_root)

nhanes_dir = os.path.join(project_root, "data", "raw", "nhanes")
os.makedirs(nhanes_dir, exist_ok=True)

from src.data.load_data import NHANES_FILES, download_nhanes_file, read_nhanes_xpt
from src.data.filters import filter_adult_cohort
from src.data.labels import compute_metabolic_syndrome_target

# 2. Load Demographic Data & Filter Adults (>= 20)
demo_path = os.path.join(nhanes_dir, "DEMO_G.XPT")
demo_df = read_nhanes_xpt(demo_path, ["SEQN", "RIDAGEYR", "RIAGENDR"])
adults_df = filter_adult_cohort(demo_df, min_age=20)
print(f"Adults >= 20 loaded: {len(adults_df)}")

# 3. Download Remaining Required Clinical XPT Files (Cycle G)
clinical_files = ["BMX", "BPX", "BPQ", "GLU", "TRIGLY", "HDL", "DIQ", "SMQ"]

for key in clinical_files:
    fname = NHANES_FILES["2011-2012"][key]
    download_nhanes_file("2011-2012", fname, output_dir=nhanes_dir)
    print(f"Verified: {fname}")

# 4. Load Specific Clinical Columns for Merging
bmx_df = read_nhanes_xpt(os.path.join(nhanes_dir, "BMX_G.XPT"), ["SEQN", "BMXWAIST"])
bpx_df = read_nhanes_xpt(os.path.join(nhanes_dir, "BPX_G.XPT"), ["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                                                                  "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"])
bpq_df = read_nhanes_xpt(os.path.join(nhanes_dir, "BPQ_G.XPT"), ["SEQN", "BPQ050A", "BPQ100D"])
glu_df = read_nhanes_xpt(os.path.join(nhanes_dir, "GLU_G.XPT"), ["SEQN", "LBXGLU", "WTSAF2YR"])
trig_df = read_nhanes_xpt(os.path.join(nhanes_dir, "TRIGLY_G.XPT"), ["SEQN", "LBXTR"])
hdl_df = read_nhanes_xpt(os.path.join(nhanes_dir, "HDL_G.XPT"), ["SEQN", "LBDHDD"])
diq_df = read_nhanes_xpt(os.path.join(nhanes_dir, "DIQ_G.XPT"), ["SEQN", "DIQ050", "DIQ070"])
smq_df = read_nhanes_xpt(os.path.join(nhanes_dir, "SMQ_G.XPT"), ["SEQN", "SMQ020", "SMQ040"])

# 5. Merge Adult Cohort with Clinical Records via SEQN
clinical_cohort = adults_df.merge(bmx_df, on="SEQN", how="inner")
clinical_cohort = clinical_cohort.merge(bpx_df, on="SEQN", how="inner")
clinical_cohort = clinical_cohort.merge(bpq_df, on="SEQN", how="left")
clinical_cohort = clinical_cohort.merge(glu_df, on="SEQN", how="inner")
clinical_cohort = clinical_cohort.merge(trig_df, on="SEQN", how="inner")
clinical_cohort = clinical_cohort.merge(hdl_df, on="SEQN", how="inner")
clinical_cohort = clinical_cohort.merge(diq_df, on="SEQN", how="left")
clinical_cohort = clinical_cohort.merge(smq_df, on="SEQN", how="left")

# 6. Compute Target and Evaluate Distribution
labeled_cohort = compute_metabolic_syndrome_target(clinical_cohort)

print(f"\nFinal Analytical Fasting Sample Size: {len(labeled_cohort)}")
print("Risk Target Distribution (Proportions):")
print(labeled_cohort["metabolic_syndrome_target"].value_counts(normalize=True).round(3))
display(labeled_cohort[["SEQN", "criteria_sum", "metabolic_syndrome_target"]].head(10))

Adults >= 20 loaded: 5560
Verified: BMX_G.XPT
Verified: BPX_G.XPT
Verified: BPQ_G.XPT
Verified: GLU_G.XPT
Verified: TRIGLY_G.XPT
Verified: HDL_G.XPT
Verified: DIQ_G.XPT
Verified: SMQ_G.XPT

Final Analytical Fasting Sample Size: 2627
Risk Target Distribution (Proportions):
metabolic_syndrome_target
0    0.582
1    0.418
Name: proportion, dtype: float64


,SEQN,criteria_sum,metabolic_syndrome_target
0,62161.0,0,0
1,62164.0,1,0
2,62169.0,1,0
3,62172.0,2,0
4,62177.0,1,0
5,62178.0,2,0
6,62184.0,0,0
7,62189.0,1,0
8,62191.0,3,1
9,62195.0,0,0


In [16]:
import os
import pyreadstat
import pandas as pd

# Resolve project root
current_dir = os.path.abspath("")
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "notebooks" else current_dir

raw_nhanes_dir = os.path.join(project_root, "data", "raw", "nhanes")
raw_physionet_dir = os.path.join(project_root, "data", "raw", "physionet")
interim_dir = os.path.join(project_root, "data", "interim")
processed_dir = os.path.join(project_root, "data", "processed")
os.makedirs(processed_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. Re-validate Wear Compliance (36,000 seconds = 600 mins/day, >= 3 days)
# ---------------------------------------------------------
print("Step 1: Re-validating wear time compliance (600 mins = 36,000 seconds/day, >= 3 days)...")

paxday_g, _ = pyreadstat.read_xport(os.path.join(raw_nhanes_dir, "PAXDAY_G.XPT"), usecols=["SEQN", "PAXDAYD", "PAXAISMD"])
paxday_h, _ = pyreadstat.read_xport(os.path.join(raw_nhanes_dir, "PAXDAY_H.XPT"), usecols=["SEQN", "PAXDAYD", "PAXAISMD"])
paxday_all = pd.concat([paxday_g, paxday_h], ignore_index=True)

# Filter valid wear days (>= 36,000 seconds = 600 minutes awake wear)
valid_days = paxday_all[paxday_all["PAXAISMD"] >= 36000.0].copy()
valid_days["daily_wear_mins"] = valid_days["PAXAISMD"] / 60.0

wear_summary = valid_days.groupby("SEQN").agg(
    valid_days_count=("PAXDAYD", "count"),
    mean_daily_wear_mins=("daily_wear_mins", "mean")
).reset_index()

wear_summary = wear_summary[wear_summary["valid_days_count"] >= 3].copy()
wear_summary["mean_daily_wear_mins"] = wear_summary["mean_daily_wear_mins"].round(2)

# Update existing clean_cohort_base.parquet directly (preserves smoking_status and all safe columns)
existing_clean_path = os.path.join(processed_dir, "clean_cohort_base.parquet")
base_df = pd.read_parquet(existing_clean_path).drop(columns=["valid_days_count", "mean_daily_wear_mins"], errors="ignore")

clean_cohort_df = pd.merge(base_df, wear_summary, on="SEQN", how="inner")
clean_cohort_df.to_parquet(existing_clean_path, index=False)

# Also filter quarantined_clinical_biomarkers.parquet to match the exact final SEQNs
quarantine_path = os.path.join(interim_dir, "quarantined_clinical_biomarkers.parquet")
if os.path.exists(quarantine_path):
    q_df = pd.read_parquet(quarantine_path)
    q_df = q_df[q_df["SEQN"].isin(clean_cohort_df["SEQN"])].copy()
    q_df.to_parquet(quarantine_path, index=False)

valid_seqns = set(clean_cohort_df["SEQN"].dropna().unique())
print(f"\nFinal Verified Cohort Size: {len(valid_seqns)} participants")
print("Mean Daily Wear Mins Summary:")
print(clean_cohort_df["mean_daily_wear_mins"].describe().round(2))
print("\nTarget Breakdown:")
print(clean_cohort_df["metabolic_syndrome_target"].value_counts())

# ---------------------------------------------------------
# 2. Chunked Filtering of PAXMIN_G & PAXMIN_H
# ---------------------------------------------------------
def filter_paxmin_chunked(xpt_path, output_path, cohort_seqns, chunk_size=250_000):
    print(f"\nFiltering {os.path.basename(xpt_path)}...")
    chunks = []
    offset = 0
    total_kept = 0
    while True:
        df_chunk, _ = pyreadstat.read_xport(xpt_path, row_offset=offset, row_limit=chunk_size)
        if df_chunk.empty:
            break
        filtered = df_chunk[df_chunk["SEQN"].isin(cohort_seqns)]
        if not filtered.empty:
            chunks.append(filtered)
            total_kept += len(filtered)
        offset += chunk_size
        print(f"  Processed {offset:,} rows | Kept {total_kept:,} rows...", end="\r")
    
    final_df = pd.concat(chunks, ignore_index=True)
    final_df.to_parquet(output_path, index=False)
    print(f"\nSaved {len(final_df):,} rows to {output_path}")

paxmin_g_file = next((os.path.join(raw_physionet_dir, f) for f in os.listdir(raw_physionet_dir) if f.lower() == "paxmin_g.xpt"), None)
paxmin_h_file = next((os.path.join(raw_physionet_dir, f) for f in os.listdir(raw_physionet_dir) if f.lower() == "paxmin_h.xpt"), None)

if paxmin_g_file and paxmin_h_file:
    filter_paxmin_chunked(paxmin_g_file, os.path.join(processed_dir, "minute_streams_cycle_g.parquet"), valid_seqns)
    filter_paxmin_chunked(paxmin_h_file, os.path.join(processed_dir, "minute_streams_cycle_h.parquet"), valid_seqns)
    print("\nALL DONE! Member 1 data processing is 100% complete.")
else:
    print(f"ERROR: Missing PAXMIN files in {raw_physionet_dir}")

Step 1: Re-validating wear time compliance (600 mins = 36,000 seconds/day, >= 3 days)...

Final Verified Cohort Size: 4603 participants
Mean Daily Wear Mins Summary:
count      4603.00
mean      45912.45
std       11808.96
min        8657.68
25%       38634.54
50%       43897.72
75%       50299.94
max      101738.06
Name: mean_daily_wear_mins, dtype: float64

Target Breakdown:
metabolic_syndrome_target
0    2618
1    1985
Name: count, dtype: int64
ERROR: Missing PAXMIN files in e:\machine learning\project\mechine-learning-\data\raw\physionet


In [18]:
import os
import pandas as pd

current_dir = os.path.abspath("")
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "notebooks" else current_dir

raw_physionet_dir = os.path.join(project_root, "data", "raw", "physionet")
processed_dir = os.path.join(project_root, "data", "processed")

clean_cohort_df = pd.read_parquet(os.path.join(processed_dir, "clean_cohort_base.parquet"))
valid_seqns = set(clean_cohort_df["SEQN"].dropna().unique())
print(f"Filtering PAXMIN files for {len(valid_seqns)} participants (Fast Streaming Mode)...")

# Only load essential accelerometer columns needed for Member 2 feature engineering
keep_cols = [
    "SEQN", "PAXDAYM", "PAXDAYWM", "PAXSSNMP",
    "PAXMTSM", "PAXTRANM", "PAXPREDM", "PAXFLGSM"
]

def fast_filter_paxmin(xpt_path, output_path, cohort_seqns, chunk_size=1_000_000):
    print(f"\nStreaming {os.path.basename(xpt_path)}...")
    chunks = []
    total_read = 0
    total_kept = 0
    
    reader = pd.read_sas(xpt_path, format="xport", chunksize=chunk_size, iterator=True)
    for df_chunk in reader:
        total_read += len(df_chunk)
        cols_present = [c for c in keep_cols if c in df_chunk.columns]
        filtered = df_chunk.loc[df_chunk["SEQN"].isin(cohort_seqns), cols_present]
        if not filtered.empty:
            chunks.append(filtered)
            total_kept += len(filtered)
        print(f"  Processed {total_read:,} rows | Kept {total_kept:,} rows...", end="\r")
    
    final_df = pd.concat(chunks, ignore_index=True)
    final_df.to_parquet(output_path, index=False)
    print(f"\nSaved {len(final_df):,} rows to {output_path}")

paxmin_g_file = next((os.path.join(raw_physionet_dir, f) for f in os.listdir(raw_physionet_dir) if f.lower() == "paxmin_g.xpt"), None)
paxmin_h_file = next((os.path.join(raw_physionet_dir, f) for f in os.listdir(raw_physionet_dir) if f.lower() == "paxmin_h.xpt"), None)

fast_filter_paxmin(paxmin_g_file, os.path.join(processed_dir, "minute_streams_cycle_g.parquet"), valid_seqns)
fast_filter_paxmin(paxmin_h_file, os.path.join(processed_dir, "minute_streams_cycle_h.parquet"), valid_seqns)
print("\nALL DONE! Member 1 data processing is 100% complete.")

Filtering PAXMIN files for 4603 participants (Fast Streaming Mode)...

Streaming PAXMIN_G.xpt...


C:\Users\MSI-G\AppData\Local\Temp\ipykernel_25916\4240094851.py:26: Pandas4Warning: The default value of 'encoding' in read_sas is deprecated. In a future version the default will change from None to 'infer', and character columns will be decoded rather than returned as bytes (XPORT files record no encoding, so 'infer' means 'ISO-8859-1'). Pass encoding='infer' to adopt the future behavior, or encoding=None to keep the current one.
  reader = pd.read_sas(xpt_path, format="xport", chunksize=chunk_size, iterator=True)


  Processed 78,126,856 rows | Kept 26,256,025 rows...
Saved 26,256,025 rows to e:\machine learning\project\mechine-learning-\data\processed\minute_streams_cycle_g.parquet

Streaming PAXMIN_H.xpt...


C:\Users\MSI-G\AppData\Local\Temp\ipykernel_25916\4240094851.py:26: Pandas4Warning: The default value of 'encoding' in read_sas is deprecated. In a future version the default will change from None to 'infer', and character columns will be decoded rather than returned as bytes (XPORT files record no encoding, so 'infer' means 'ISO-8859-1'). Pass encoding='infer' to adopt the future behavior, or encoding=None to keep the current one.
  reader = pd.read_sas(xpt_path, format="xport", chunksize=chunk_size, iterator=True)


  Processed 88,223,479 rows | Kept 26,242,498 rows...
Saved 26,242,498 rows to e:\machine learning\project\mechine-learning-\data\processed\minute_streams_cycle_h.parquet

ALL DONE! Member 1 data processing is 100% complete.


In [19]:
import pandas as pd

# Load the processed Parquet files for Cycle G and Cycle H
df_g = pd.read_parquet(r"e:\machine learning\project\mechine-learning-\data\processed\minute_streams_cycle_g.parquet")
df_h = pd.read_parquet(r"e:\machine learning\project\mechine-learning-\data\processed\minute_streams_cycle_h.parquet")

# Verify total unique participants across both cycles
unique_seqn = set(df_g["SEQN"].unique()).union(set(df_h["SEQN"].unique()))

print(f"Cycle G shape: {df_g.shape} | Cycle H shape: {df_h.shape}")
print(f"Total unique participants (SEQN): {len(unique_seqn)}")
print("\nCycle G null values:\n", df_g.isna().sum())
print("\nCycle H null values:\n", df_h.isna().sum())

Cycle G shape: (26256025, 8) | Cycle H shape: (26242498, 8)
Total unique participants (SEQN): 4603

Cycle G null values:
 SEQN        0
PAXDAYM     0
PAXDAYWM    0
PAXSSNMP    0
PAXMTSM     0
PAXTRANM    0
PAXPREDM    0
PAXFLGSM    0
dtype: int64

Cycle H null values:
 SEQN        0
PAXDAYM     0
PAXDAYWM    0
PAXSSNMP    0
PAXMTSM     0
PAXTRANM    0
PAXPREDM    0
PAXFLGSM    0
dtype: int64
